## Deep Research Agent ##

For Deep research agent, a reasoning model is paired with external tools like web search to retrieve internet search data. This agent will follow the ReAct pattern (Reason and Action). The model thinks, comes up with observations and continues to reason till it come up with final answer or reaches a step limit.

For a multi-agent model, instead of single agent, we can design multiple collaborating agents that work in parallel.
1. Planner: Analyzes the query and breaks it into sub-questions
2. Researchers: Runs in parallel, each searching and summarizing findings for one sub-question
3. Synthesizer: Combines and summarized all research into a final report

In [ ]:
import asyncio
from openai import AsyncOpenAI
from ddgs import DDGS

client = AsyncOpenAI(api_key="ollama", base_url="http://localhost:11434/v1")
MODEL = "llama3.2:3b"


async def plan_research(query: str) -> list[str]:
    """Planner agent: breaks query into sub-questions and decides scale (1, 3, or up to 5 sub-queries)."""
    prompt = f"""You are a research planner. Given a query, break it into 1-5 focused sub-questions.
            - Simple factual queries: 1 sub-question
            - Moderate topics: 3 sub-questions
            - Complex topics needing multiple angles: 5 sub-questions

            Query: {query}

            Return ONLY the sub-questions, one per line, no numbering or bullets."""

    r = await client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    lines = [line.strip() for line in r.choices[0].message.content.strip().split("\n") if line.strip()]
    return lines[:5]  # cap at 5


async def search_and_summarize(sub_question: str) -> dict:
    """Researcher agent: searches web and summarizes findings for one sub-question."""
    with DDGS() as ddgs:
        results = [hit["body"] for hit in ddgs.text(sub_question, max_results=3)]
    snippets = "\n".join(results)

    prompt = f"""Based on these search results, write a concise summary answering: {sub_question}

            Search results:
            {snippets}

            Summary:"""

    r = await client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    return {"question": sub_question, "summary": r.choices[0].message.content.strip()}


async def synthesize_report(query: str, findings: list[dict]) -> str:
    """Synthesizer agent: combines all findings into a coherent report."""
    findings_text = "\n\n".join([f"### {f['question']}\n{f['summary']}" for f in findings])

    prompt = f"""You are a research synthesizer. Combine these findings into a coherent report.

            Original query: {query}

            Research findings:
            {findings_text}

            Write a well-structured report that answers the original query. Use markdown formatting."""

    r = await client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.4
    )
    return r.choices[0].message.content.strip()


async def deep_research(query: str) -> str:
    """Run the full multi-agent deep research pipeline."""
    print(f"Planning research for: {query}\n")

    # Step 1: Plan
    sub_questions = await plan_research(query)
    print(f"Sub-questions ({len(sub_questions)}):")
    for sq in sub_questions:
        print(f"  - {sq}")

    # Step 2: Research in parallel
    print("\nResearching in parallel...")
    findings = list(await asyncio.gather(*[search_and_summarize(sq) for sq in sub_questions]))
    print(f"Collected {len(findings)} research summaries.")

    # Step 3: Synthesize
    print("\nSynthesizing final report...\n")
    report = await synthesize_report(query, findings)
    return report


# Run the multi-agent research
query = "What are the best resources to learn machine learning in 2025?"
report = await deep_research(query)
print("=" * 60)
print("FINAL REPORT")
print("=" * 60)
print(report)